# R11-H101 - The identity benchmark (25 pairs cannot carry the layer)

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R11 identity round, benchmark construction + local-model adjudication <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only); adjudicator gpt-oss-120b @ localhost:8010 <br>

Builds a 200+ pair labeled identity benchmark. Candidates from ALL detector families: the 66-pair
forensic inventory, the 127 SAME_AS edges (incl 6 labeled false merges), model-number sibling pairs,
transitive SAME_AS chain pairs (alias-chain variance), and random distractors. Every pair is adjudicated
by the local model against source records (name, types, description, spec lines, source docs) with a
YES/NO/UNCERTAIN verdict and a one-line evidence string. Then: (a) resolver-proxy metrics on the new
labels vs the 25-pair figures (bar: >=10-point shift on a headline metric confirms); (b) sanity check on
the 6 known false merges + the P10 mask/battery class (the model should get these right).

In [1]:
import json, datetime, glob, random, re, time, collections
from concurrent.futures import ThreadPoolExecutor
from neo4j import GraphDatabase
from openai import OpenAI
from rich import print as rprint
NEO4J_URI='bolt://user-konrad.jelen-kgf-neo4j2:7687'
random.seed(11)
client=OpenAI(base_url='http://localhost:8010/v1', api_key='x', timeout=180.0)
MODEL='gpt-oss-120b'

## Pull entity records

In [2]:
drv=GraphDatabase.driver(NEO4J_URI, auth=('neo4j','kgfoundry'), notifications_min_severity='OFF')
with drv.session() as s:
    ents=s.run('MATCH (e:Entity) RETURN e AS e, labels(e) AS types, e.id AS id').data()
    docmap={r['id']:r['name'] for r in s.run('MATCH (k:KGFDocument) RETURN k.id AS id, k.name AS name')}
    sa_pairs_graph=s.run('MATCH (a:Entity)-[:SAME_AS]-(b:Entity) WHERE a.id<b.id RETURN a.name AS a, b.name AS b').data()
drv.close()
by_name={}
rec={}
for r in ents:
    e=r['e']; nm=e['name']
    by_name.setdefault(nm.lower(), nm)
    specs=[(k[5:],v) for k,v in e.items() if k.startswith('prop_') and v not in (None,'')]
    rec[nm.lower()]=dict(name=nm, types=[t for t in r['types'] if t!='Entity'] or ['Entity'],
        description=(e.get('description') or '')[:400],
        specs=specs[:10],
        docs=[docmap.get(d,d) for d in (e.get('source_documents') or [])][:3])
rprint(f'{len(ents)} entities, {len(by_name)} distinct names')

# SAME_AS adjacency for transitive chains
adj=collections.defaultdict(set)
for p in sa_pairs_graph:
    adj[p['a']].add(p['b']); adj[p['b']].add(p['a'])
# connected components
seen=set(); comps=[]
for n in adj:
    if n in seen: continue
    stack=[n]; comp=set()
    while stack:
        x=stack.pop()
        if x in seen: continue
        seen.add(x); comp.add(x); stack+=[y for y in adj[x] if y not in seen]
    if len(comp)>=3: comps.append(sorted(comp))
rprint(f'SAME_AS components size>=3 (alias chains): {len(comps)}; sizes {sorted((len(c) for c in comps),reverse=True)[:10]}')

2798 entities, 2796 distinct names

SAME_AS components size>=3 (alias chains): 18; sizes [36, 9, 7, 7, 6, 5, 5, 5, 4, 4]

## Assemble candidate pairs from all detector families

In [3]:
ff=json.load(open(sorted(glob.glob('../reports/identity-forensics-r11-final-*.json'))[-1]))
mr=json.load(open(sorted(glob.glob('../reports/matching-r12-foundation-*.json'))[-1]))
tp=json.load(open(sorted(glob.glob('../reports/topology-r10-*.json'))[-1]))

false_set=set(frozenset((x['a'].lower(),x['b'].lower())) for x in ff['labeled_false_merges'])
cand={}  # frozenset(names lower) -> dict(a,b,source)
def add(a,b,src):
    if a.lower() not in rec or b.lower() not in rec: return
    k=frozenset((a.lower(),b.lower()))
    if len(k)<2: return
    cand.setdefault(k, dict(a=rec[a.lower()]['name'], b=rec[b.lower()]['name'], sources=[]))['sources'].append(src)

for p in ff['pairs']: add(p['a'],p['b'],'inventory')                      # 66 forensic inventory
for e in tp['sa_edges']: add(e['a'],e['b'],'same_as_edge')               # 127 SAME_AS
for p in mr['pairs']:
    if p['cls']=='sibling': add(p['a'],p['b'],'sibling')                 # model-number siblings
# transitive alias-chain pairs (non-direct edges within components)
direct=set(frozenset((p['a'].lower(),p['b'].lower())) for p in sa_pairs_graph)
chain_added=0
for comp in comps:
    for i in range(len(comp)):
        for j in range(i+1,len(comp)):
            k=frozenset((comp[i].lower(),comp[j].lower()))
            if k not in direct and chain_added<30:
                add(comp[i],comp[j],'alias_chain_transitive'); chain_added+=1
# random distractors
allnames=list(by_name.values())
radd=0
while radd<40:
    a,b=random.sample(allnames,2)
    k=frozenset((a.lower(),b.lower()))
    if k not in cand:
        add(a,b,'random'); radd+=1
# mark labeled-false
for k,v in cand.items():
    v['labeled_false']=k in false_set
rprint(f'[bold]candidate pairs: {len(cand)}[/bold]')
srccount=collections.Counter(v['sources'][0] for v in cand.values())
rprint('by primary source:', dict(srccount))
rprint('labeled-false pairs present:', sum(v['labeled_false'] for v in cand.values()))

candidate pairs: 298

by primary source:
{'inventory': 66, 'same_as_edge': 125, 'sibling': 38, 'alias_chain_transitive': 29, 'random': 40}

labeled-false pairs present: 6

## Adjudicate every pair with the local model (concurrency 2)

In [4]:
def render(name):
    r=rec[name.lower()]
    sp='; '.join(f'{k}={v}' for k,v in r['specs']) or '(none)'
    return f"name: {r['name']}\ntypes: {', '.join(r['types'])}\ndescription: {r['description'] or '(none)'}\nspecs: {sp}\nsource_docs: {', '.join(r['docs']) or '(none)'}"

PROMPT=("You judge whether two knowledge-graph records refer to the SAME real-world item "
  "(same physical product/model/component/concept), not merely related or same-family. "
  "Different model numbers, different accessories of one device, or a device vs its part are NOT the same item.\n\n"
  "RECORD A:\n{a}\n\nRECORD B:\n{b}\n\n"
  "Answer in exactly two lines:\nVERDICT: YES or NO or UNCERTAIN\nEVIDENCE: one sentence citing a specific field.")

def adjudicate(item):
    k,v=item
    msg=PROMPT.format(a=render(v['a']), b=render(v['b']))
    for attempt in range(2):
        try:
            r=client.chat.completions.create(model=MODEL, temperature=0, max_tokens=1600,
                messages=[{'role':'user','content':msg}])
            txt=r.choices[0].message.content or ''
            m=re.search(r'VERDICT:\s*(YES|NO|UNCERTAIN)', txt, re.I)
            e=re.search(r'EVIDENCE:\s*(.+)', txt, re.I)
            verdict=(m.group(1).upper() if m else 'UNCERTAIN')
            return k, verdict, (e.group(1).strip()[:300] if e else txt.strip()[:200])
        except Exception as ex:
            if attempt==1: return k,'ERROR',str(ex)[:200]
            time.sleep(3)

items=list(cand.items())
results={}
t0=time.time()
with ThreadPoolExecutor(max_workers=2) as ex:
    for i,(k,verdict,ev) in enumerate(ex.map(adjudicate, items)):
        results[k]=(verdict,ev)
        if (i+1)%25==0: rprint(f'  adjudicated {i+1}/{len(items)}  ({time.time()-t0:.0f}s)')
rprint(f'[bold]done {len(results)} pairs in {time.time()-t0:.0f}s[/bold]')
rprint('verdict distribution:', dict(collections.Counter(v for v,_ in results.values())))

adjudicated 25/298  (44s)

adjudicated 50/298  (97s)

adjudicated 75/298  (143s)

adjudicated 100/298  (193s)

adjudicated 125/298  (240s)

...

adjudicated 225/298  (461s)

adjudicated 250/298  (505s)

adjudicated 275/298  (547s)

done 298 pairs in 585s

verdict distribution:
{'YES': 52, 'NO': 245, 'UNCERTAIN': 1}

## Build benchmark records + difficulty tiers

In [5]:
def tier(v):
    s=set(v['sources'])
    if v['labeled_false']: return 'known_false'
    if 'sibling' in s: return 'sibling'
    if 'alias_chain_transitive' in s: return 'alias_chain'
    if 'random' in s: return 'distractor'
    if 'same_as_edge' in s: return 'alias_edge'
    return 'inventory_variance'

bench=[]
for k,v in cand.items():
    verdict,ev=results[k]
    ra,rb=rec[v['a'].lower()], rec[v['b'].lower()]
    bench.append(dict(a=v['a'], b=v['b'], a_id=None, b_id=None,
        a_types=ra['types'], b_types=rb['types'],
        a_docs=ra['docs'], b_docs=rb['docs'],
        detector_sources=sorted(set(v['sources'])), tier=tier(v),
        labeled_false=v['labeled_false'],
        model_verdict=verdict, evidence=ev))
tiers=collections.Counter(b['tier'] for b in bench)
rprint('difficulty tiers:', dict(tiers))
rprint('label distribution (model):', dict(collections.Counter(b['model_verdict'] for b in bench)))

difficulty tiers:
{'inventory_variance': 64, 'sibling': 40, 'alias_edge': 119, 'known_false': 6, 'alias_chain': 29, 'distractor': 40}

label distribution (model):
{'YES': 52, 'NO': 245, 'UNCERTAIN': 1}

## Resolver-proxy metrics vs the 25-pair figures

In [6]:
# resolver prediction: SAME_AS edge (or alias_edge source) => resolver MERGED (pred=1); else pred=0
def resolver_pred(b):
    return 1 if ('same_as_edge' in b['detector_sources'] or 'alias_chain_transitive' in b['detector_sources']) else 0
# label from model: YES=1 same, NO=0 different; drop UNCERTAIN/ERROR
scored=[b for b in bench if b['model_verdict'] in ('YES','NO')]
y=[1 if b['model_verdict']=='YES' else 0 for b in scored]
pred=[resolver_pred(b) for b in scored]
n=len(scored)
tp_=sum(1 for a,c in zip(pred,y) if a==1 and c==1)
fp_=sum(1 for a,c in zip(pred,y) if a==1 and c==0)
fn_=sum(1 for a,c in zip(pred,y) if a==0 and c==1)
tn_=sum(1 for a,c in zip(pred,y) if a==0 and c==0)
acc=(tp_+tn_)/n
prec=tp_/(tp_+fp_) if (tp_+fp_) else 0.0
rec_=tp_/(tp_+fn_) if (tp_+fn_) else 0.0
f1=2*prec*rec_/(prec+rec_) if (prec+rec_) else 0.0
BASE_ACC=0.44  # 25-pair calibration accuracy (registration / memory)
rprint(f'scored pairs (YES/NO): {n}  | confusion tp={tp_} fp={fp_} fn={fn_} tn={tn_}')
rprint(f'[bold]resolver-proxy accuracy {acc:.1%} (25-pair baseline {BASE_ACC:.0%}, shift {abs(acc-BASE_ACC)*100:.0f} pts)[/bold]')
rprint(f'precision {prec:.1%}  recall {rec_:.1%}  F1 {f1:.1%}')

scored pairs (YES/NO): 297  | confusion tp=22 fp=133 fn=30 tn=112

resolver-proxy accuracy 45.1% (25-pair baseline 44%, shift 1 pts)

precision 14.2%  recall 42.3%  F1 21.3%

## Sanity check - known false merges + P10 mask/battery

In [7]:
known=[b for b in bench if b['labeled_false']]
p10=[b for b in bench if re.search(r'P10', b['a']+' '+b['b']) and re.search(r'batter|transcend', (b['a']+' '+b['b']), re.I)]
def right_no(b): return b['model_verdict']=='NO'  # known false => correct answer is NO (different)
known_acc=sum(right_no(b) for b in known)/len(known) if known else None
rprint(f'known false merges in benchmark: {len(known)}; model correct (NO): {sum(right_no(b) for b in known)}/{len(known)}')
for b in known: rprint(f"  [{b['model_verdict']}] {b['a']!r} vs {b['b']!r} :: {b['evidence'][:100]}")
rprint(f'P10 mask/battery class pairs: {len(p10)}')
for b in p10: rprint(f"  [{b['model_verdict']}] {b['a']!r} vs {b['b']!r} :: {b['evidence'][:100]}")
sanity_pool=known+[b for b in p10 if b not in known]
sanity_acc=sum(b['model_verdict']=='NO' for b in sanity_pool)/len(sanity_pool) if sanity_pool else None
rprint(f'[bold]adjudicator sanity accuracy on known-NO class: {sanity_acc:.0%}[/bold]' if sanity_acc is not None else 'no sanity pairs')

known false merges in benchmark: 6; model correct (NO): 6/6

[NO] 'Body Position Sensor Sandman SD20' vs 'BreathSensor cable (SD20, adult, 220 connector, 3-pin keyhole)' :: 
Record A has part_number P1719 and a 249 (2‑pin) connector, while Record B has part_number P1112 and

[NO] 'Sensor adapter WristOx2 3150' vs 'USB PC download cable WristOx2' :: Record A is an 8‑to‑9‑pin sensor 
adapter (part_number 1084509) while Record B is a USB PC download c

[NO] 'Sensor adapter WristOx2 3150' vs 'USB PC download cable for WristOx2' :: Record A is an 8‑to‑9 pin adapter 
(part_number 1084509) while Record B is a USB download cable (part

[NO] 'Sensor adapter WristOx2 3150' vs 'WristOx2 wrist band 10 inch' :: Record A is an “Adapter converting 8 pin 
to 9 pin for WristOx2 3150,” while Record B is a “Reusable

...

P10 mask/battery class pairs: 3

[NO] 'AirFit P10' vs 'Transcend P10 battery' :: Record A is a CPAPDevice mask (AirFit P10) while Record B is an 
Accessory (Transcend P10 battery), i

[NO] 'AirFit P10 bedside starter kit' vs 'Transcend P10 battery' :: Record A is a “AirFit P10 bedside starter 
kit” (ProductModel) while Record B is a “Transcend P10 bat

[NO] 'AirFit P10 for AirMini' vs 'Transcend P10 battery' :: Record A specifies a nasal pillow mask 
(mask_type=nasal pillow, part_number=380023) while Record B s

adjudicator sanity accuracy on known-NO class: 100%

## Save benchmark + verdict

In [8]:
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
shift=abs(acc-BASE_ACC)*100
confirm = len(bench)>=200 and shift>=10
adjudication_noise = (sanity_acc is not None and sanity_acc<0.5)
verdict = 'CONFIRMED' if confirm else ('REFUTED' if len(bench)>=200 else 'INCOMPLETE')
reason=(f'{len(bench)}-pair benchmark shipped; resolver-proxy accuracy {acc:.1%} vs 25-pair 44% = {shift:.0f}-pt shift '
        f'({"confirms" if shift>=10 else "within 5-10pt, does not confirm"} the noise-domination claim)')
if adjudication_noise: reason+=' | WARNING: adjudicator failed sanity class (<50% on known-NO) - metric shift may be adjudication noise'
report=dict(hypothesis='R11-H101',
  benchmark_size=len(bench), scored_pairs=n,
  tiers=dict(tiers), detector_source_counts=dict(collections.Counter(s for b in bench for s in b['detector_sources'])),
  model_verdict_distribution=dict(collections.Counter(b['model_verdict'] for b in bench)),
  resolver_proxy=dict(accuracy=acc, precision=prec, recall=rec_, f1=f1, tp=tp_,fp=fp_,fn=fn_,tn=tn_),
  baseline_25pair_accuracy=BASE_ACC, headline_shift_pts=shift,
  known_false_count=len(known), known_false_model_correct=sum(right_no(b) for b in known),
  p10_class=[dict(a=b['a'],b=b['b'],verdict=b['model_verdict']) for b in p10],
  adjudicator_sanity_accuracy=sanity_acc, adjudication_noise_flag=adjudication_noise,
  bar='benchmark shipped + >=10-pt shift on a headline metric confirms; within 5pts refutes',
  verdict=verdict, reason=reason,
  pairs=bench)
path=f'../reports/identity-benchmark-h101-{stamp}.json'
json.dump(report, open(path,'w'), indent=2)
rprint(f'[bold green]{verdict}[/bold green] - {reason}')
rprint('wrote', path)

REFUTED - 298-pair benchmark shipped; resolver-proxy accuracy 45.1% vs 25-pair 44% = 1-pt shift (within 5-10pt, 
does not confirm the noise-domination claim)

wrote ../reports/identity-benchmark-h101-20260707-094448.json